In [6]:
import requests
import pandas as pd

# Define Chicago coordinates and the long-term date range
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 41.85,
    "longitude": -87.65,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_mean,precipitation_sum"
}

response = requests.get(url, params=params)
data = response.json()

# Convert the results into a readable table
df = pd.DataFrame(data['daily'])
print(df.head())

         time  temperature_2m_mean  precipitation_sum
0  2022-01-01                  2.7                5.5
1  2022-01-02                 -3.2                7.0
2  2022-01-03                -11.3                0.0
3  2022-01-04                 -4.4                0.0
4  2022-01-05                 -5.2                0.1


In [5]:
import requests
import pandas as pd

# Define Chicago coordinates and the long-term date range
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 41.85,
    "longitude": -87.65,
    "start_date": "1990-01-01",
    "end_date": "2023-12-31",
    "hourly": "temperature_2m,precipitation"
}

response = requests.get(url, params=params)
data = response.json()

# Convert the results into a readable table
df = pd.DataFrame(data["hourly"])

# Convert time column to datetime (optional but useful)
df["time"] = pd.to_datetime(df["time"])

# Save to CSV
df.to_csv("chicago_weather_1990_2023_hourly.csv", index=False)

print(df.head())
print("Rows:", len(df))

                 time  temperature_2m  precipitation
0 1990-01-01 00:00:00            -1.5            0.0
1 1990-01-01 01:00:00            -1.6            0.0
2 1990-01-01 02:00:00            -1.7            0.0
3 1990-01-01 03:00:00            -1.8            0.0
4 1990-01-01 04:00:00            -1.9            0.0
Rows: 298032


In [8]:
#Add new column header 
import pandas as pd 
import requests

csv_file_path = "weather_crime.csv"
crime_df = pd.read_csv(csv_file_path)

# Make sure these columns exist
crime_df["Daily Temperature Mean"] = ""
crime_df["Daily Precipitation Sum"] = ""

#Create map for coordinates and cities 
crime_df["date"] = pd.to_datetime(crime_df["date"]).dt.date.astype(str)

# Correct city -> (latitude, longitude)
map_coord = {
    "Chicago": (41.85, -87.65),
    "NYC": (40.71, -74.00),
    "Durham": (35.99, -78.90),
    "Charlottesville": (38.03, -78.47)
}

weather_frames = []

url = "https://archive-api.open-meteo.com/v1/archive"

for city, (lat, lon) in map_coord.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2022-01-01",
        "end_date": "2025-12-31",
        "daily": "temperature_2m_mean,precipitation_sum",
        "timezone": "America/New_York"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    city_weather = pd.DataFrame(data["daily"])
    city_weather["city"] = city

    # Rename columns to match your CSV headers
    city_weather = city_weather.rename(columns={
        "time": "date",
        "temperature_2m_mean": "Daily Temperature Mean",
        "precipitation_sum": "Daily Precipitation Sum"
    })

    weather_frames.append(city_weather)

# Combine all city weather data
weather_df = pd.concat(weather_frames, ignore_index=True)

# Merge into crime dataset using city + date
merged_df = crime_df.drop(columns=["Daily Temperature Mean", "Daily Precipitation Sum"], errors="ignore").merge(
    weather_df[["date", "city", "Daily Temperature Mean", "Daily Precipitation Sum"]],
    on=["date", "city"],
    how="left"
)

# Save final file
merged_df.to_csv("weather_crime_final.csv", index=False)

print(merged_df.head())
print("Saved to weather_crime_final.csv")

#Add each cities data accordingly

              city        date  total_crime_count  violent_count  \
0  Charlottesville  2022-01-01                 15              0   
1  Charlottesville  2022-01-02                 11              0   
2  Charlottesville  2022-01-03                  8              0   
3  Charlottesville  2022-01-04                 16              0   
4  Charlottesville  2022-01-05                 12              0   

   property_count  other_count  Daily Temperature Mean  \
0               2           13                    14.7   
1               2            9                    15.4   
2               1            7                     0.9   
3               1           15                    -6.4   
4               0           12                     0.2   

   Daily Precipitation Sum  
0                      9.0  
1                     16.9  
2                     29.4  
3                      0.0  
4                      0.0  
Saved to weather_crime_final.csv
